In [1]:
import re, json, math, ast, io, operator, contextlib
from collections import deque, Counter
from datetime import date

# ---- settings you can change ----
BUFFER_SIZE   = 4      # how many recent turns short-term memory keeps
RETRIEVAL_MIN = 0.35   # minimum cosine similarity to trust a retrieved memory (below this the agent says "I don't know")
RECENCY_WINDOW = 0.15  # if two memories score within this gap, the newer one wins (handles updated facts)
SEARCH_MIN    = 0.20   # same idea for the search tool
TODAY = date(2026, 12, 1)   # fixed demo date so results are repeatable

STOP = set("a an the is am are was what when do does did i my me on of to in how many per s it that and for at will be there can you your".split())

def tokens(text):
    out = []
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        if w in STOP:
            continue
        if len(w) > 3 and w.endswith("s"):
            w = w[:-1]              # tiny stemmer: hours -> hour, days -> day
        out.append(w)
    return out

def rank(query, entries):
    """entries: list of (text, seq). Returns [(score, text, seq)] best first. TF-IDF + cosine, written by hand."""
    docs = [Counter(tokens(t)) for t, _ in entries]
    q = Counter(tokens(query))
    n = len(docs)
    df = Counter(w for d in docs for w in d)
    idf = lambda w: math.log((1 + n) / (1 + df[w])) + 1
    def vec(c): return {w: tf * idf(w) for w, tf in c.items()}
    qv = vec(q)
    qn = math.sqrt(sum(v * v for v in qv.values())) or 1
    out = []
    for (text, seq), d in zip(entries, docs):
        dv = vec(d)
        dn = math.sqrt(sum(v * v for v in dv.values())) or 1
        dot = sum(qv[w] * dv.get(w, 0) for w in qv)
        out.append((round(dot / (qn * dn), 3), text, seq))
    return sorted(out, reverse=True)

class VectorStore:
    """Long-term memory. Stores (text, seq) and retrieves by similarity. Can be saved and loaded as JSON."""
    def __init__(self):
        self.items = []
    def add(self, text, seq):
        self.items.append((text, seq))
    def search(self, query, k=3):
        return rank(query, self.items)[:k]
    def to_json(self):
        return json.dumps(self.items)
    @classmethod
    def from_json(cls, s):
        v = cls(); v.items = [tuple(x) for x in json.loads(s)]
        return v

In [2]:
# ---- calculator (safe: only numbers and + - * / **) ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def calculator(expr):
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        raise ValueError("not allowed")
    v = ev(ast.parse(expr.strip(), mode="eval").body)
    return int(v) if float(v).is_integer() else round(v, 4)

# ---- code execution (restricted demo sandbox, NOT a real security boundary) ----
SAFE_BUILTINS = {k: __builtins__[k] if isinstance(__builtins__, dict) else getattr(__builtins__, k)
                 for k in ("print", "range", "sum", "len", "min", "max", "sorted", "abs", "round",
                           "list", "int", "float", "str", "enumerate", "zip")}
def run_code(code):
    # tool-call check: reject risky code BEFORE running it
    if len(code) > 200:
        return "BLOCKED: code too long"
    if re.search(r"\b(import|open|exec|eval|compile|input|globals|locals|getattr|while|os|sys)\b|__|\*\*|\d{7,}", code):
        return "BLOCKED: code uses something not allowed in this sandbox"
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {"__builtins__": SAFE_BUILTINS}, {})
    except Exception as e:
        return f"ERROR: {type(e).__name__}"
    return buf.getvalue().strip() or "(no output)"

# ---- search tool over a small local corpus (stands in for a web search API) ----
CORPUS = [
    "Overfitting happens when a model learns the training data too well, including its noise, and then performs poorly on new data.",
    "Precision is the fraction of predicted positives that are actually positive. Recall is the fraction of actual positives that were found.",
    "A vector database stores embeddings and finds the closest ones to a query using a similarity measure such as cosine similarity.",
    "ReAct is a pattern where an agent alternates between reasoning and acting, using the result of each action to decide the next step.",
    "Short-term memory holds the recent context of one conversation. Long-term memory keeps information across sessions.",
    "A confusion matrix counts true positives, false positives, true negatives and false negatives for a classifier.",
]
def search_tool(query):
    best = rank(query, [(t, i) for i, t in enumerate(CORPUS)])[0]
    return best[1] if best[0] >= SEARCH_MIN else None

# ---- date tool ----
MONTHS = ["january","february","march","april","may","june","july","august","september","october","november","december"]
def days_until(text):
    m = re.search(r"(\d{1,2})\s+(" + "|".join(MONTHS) + ")", text.lower())
    if not m:
        return None
    return (date(TODAY.year, MONTHS.index(m.group(2)) + 1, int(m.group(1))) - TODAY).days

# quick self-test of the tools
print(calculator("45 * 12"), "|", run_code("print(sum(range(1, 11)))"), "|", run_code("import os"), "|", search_tool("overfitting")[:40])

540 | 55 | BLOCKED: code uses something not allowed in this sandbox | Overfitting happens when a model learns 


In [3]:
UNKNOWN = "I don't know"

class Agent:
    def __init__(self, use_short=True, use_long=True, use_tools=True, store_json=None):
        self.use_short, self.use_long, self.use_tools = use_short, use_long, use_tools
        self.buffer = deque(maxlen=BUFFER_SIZE)                  # SHORT-TERM memory (recent turns only)
        self.store = VectorStore.from_json(store_json) if (use_long and store_json) else VectorStore()   # LONG-TERM memory
        self.seq = max([s for _, s in self.store.items], default=0)   # turn counter continues across sessions

    # ----- memory read: look in the buffer and/or the vector store -----
    def recall(self, query, used):
        cands = {}
        if self.use_short:
            for t in self.buffer:
                if t["kind"] == "stmt": cands[t["seq"]] = t["user"]
        if self.use_long:
            for text, seq in self.store.items: cands[seq] = text
        if not cands:
            return None
        ranked = rank(query, [(t, s) for s, t in cands.items()])
        best = ranked[0][0]
        if best < RETRIEVAL_MIN:
            return None
        close = [r for r in ranked if r[0] >= max(best - RECENCY_WINDOW, RETRIEVAL_MIN)]
        pick = max(close, key=lambda r: r[2])                    # newest among the near-best (handles updated facts)
        if self.use_long and any(pick[1] == t for t, _ in self.store.items): used.append("long-term memory")
        if self.use_short and any(pick[1] == t["user"] for t in self.buffer): used.append("short-term memory")
        return pick[1]

    def last_result(self):
        if not self.use_short: return None
        for t in reversed(self.buffer):
            if t.get("result") is not None: return t["result"]
        return None

    def respond(self, text):
        self.seq += 1
        used, result = [], None
        low = text.lower().strip()
        answer = self._decide(text, low, used)
        if isinstance(answer, tuple): answer, result = answer
        kind = "stmt" if (not text.strip().endswith("?") and answer.startswith(("Noted", "Okay"))) else "q"
        if self.use_short:
            self.buffer.append({"seq": self.seq, "user": text, "answer": answer, "result": result, "kind": kind})
        if kind == "stmt" and self.use_long:
            self.store.add(text, self.seq); used.append("wrote long-term memory")
        return {"answer": answer, "used": used}

    def _decide(self, text, low, used):
        no_tool = UNKNOWN + " (no tool available for that)."
        # 1) explicit tool commands
        if low.startswith("search:"):
            if not self.use_tools: return no_tool
            used.append("tool: search")
            hit = search_tool(text[7:])
            return hit if hit else UNKNOWN + " (nothing found)."
        if low.startswith("run:"):
            if not self.use_tools: return no_tool
            used.append("tool: code")
            out = run_code(text[4:].strip())
            return (out, None)
        # 2) references to the previous result ("double that", "subtract 80 from it")
        ref = re.match(r"(?:now )?(double|triple|halve) (?:that|it)\.?$", low) or \
              re.match(r"(?:now )?(add|subtract|multiply) (\d+(?:\.\d+)?) (?:to|from|with) (?:that|it)\.?$", low)
        if ref:
            if not self.use_tools: return no_tool
            prev = self.last_result()
            if prev is None: return UNKNOWN + " what 'that' refers to."
            used.append("short-term memory"); used.append("tool: calculator")
            g = ref.groups()
            expr = {"double": f"{prev}*2", "triple": f"{prev}*3", "halve": f"{prev}/2"}.get(g[0]) or \
                   {"add": f"{prev}+{g[-1]}", "subtract": f"{prev}-{g[-1]}", "multiply": f"{prev}*{g[-1]}"}[g[0]]
            r = calculator(expr)
            return (f"{r}", r)
        # 3) plain arithmetic
        m = re.search(r"\d+(?:\.\d+)?(?:\s*[-+*/]\s*\d+(?:\.\d+)?)+", text)
        if m:
            if not self.use_tools: return no_tool
            used.append("tool: calculator")
            r = calculator(m.group(0))
            return (f"{r}", r)
        # 4) questions that combine memory and a tool
        m = re.search(r"how many hours .* in (\d+) days?", low)
        if m:
            mem = self.recall("study hours per day", used)
            if mem is None: return UNKNOWN + " (I have no study-hours fact)."
            per_day = re.search(r"\d+(?:\.\d+)?", mem)
            if not self.use_tools: return f"I recall: '{mem}'. " + no_tool
            used.append("tool: calculator")
            r = calculator(f"{per_day.group(0)}*{m.group(1)}")
            return (f"{r} hours", r)
        if re.search(r"how many days .*exam", low):
            mem = self.recall("my exam", used)
            if mem is None: return UNKNOWN + " (I have no exam date)."
            if not self.use_tools: return f"I recall: '{mem}'. " + no_tool
            used.append("tool: date")
            d = days_until(mem)
            return (f"{d} days (today is {TODAY})", d) if d is not None else UNKNOWN + " (no date in the memory)."
        # 5) personal questions -> memory
        if low.endswith("?") and re.search(r"\b(my|i|me)\b", low):
            mem = self.recall(text, used)
            return mem if mem else UNKNOWN + "."
        # 6) general questions -> search tool
        if low.endswith("?"):
            if not self.use_tools: return no_tool
            used.append("tool: search")
            hit = search_tool(text)
            return hit if hit else UNKNOWN + " (nothing found)."
        # 7) otherwise it is a statement: remember it
        return "Noted." if (self.use_short or self.use_long) else "Okay."

In [4]:
SESSION_1 = [
    "My exam is on 12 December.",
    "I study 3 hours every day.",
    "I am working on a spam filter project.",
    "My favourite subject is machine learning.",
    "What is my favourite subject?",
    "What is 45 * 12?",
    "Double that.",
    "Now subtract 80 from it.",
    "Search: what is overfitting?",
    "Run: print(sum(range(1, 11)))",
    "When is my exam?",
    "My exam moved to 15 December.",
]
SESSION_2 = [
    "Double that.",
    "When is my exam?",
    "How many hours do I study per day?",
    "What project am I working on?",
    "How many hours will I study in 10 days?",
    "How many days are left until my exam?",
    "What is my mother's phone number?",
    "When do I sit for my paper?",
    "What is precision?",
    "What is my favourite food?",
]

def show(agent, messages):
    for msg in messages:
        r = agent.respond(msg)
        tag = f"   [{', '.join(dict.fromkeys(r['used']))}]" if r["used"] else ""
        print(f"  Student: {msg}\n  Agent  : {r['answer']}{tag}\n")

full = Agent(True, True, True)
print("=== SESSION 1 ===\n"); show(full, SESSION_1)
saved = full.store.to_json()                     # what would be written to disk at the end of the day
print("(saved long-term memory:", len(json.loads(saved)), "facts)\n")
print("=== SESSION 2 (new agent: empty buffer, loads the saved store) ===\n")
full2 = Agent(True, True, True, store_json=saved); show(full2, SESSION_2)

=== SESSION 1 ===

  Student: My exam is on 12 December.
  Agent  : Noted.   [wrote long-term memory]

  Student: I study 3 hours every day.
  Agent  : Noted.   [wrote long-term memory]

  Student: I am working on a spam filter project.
  Agent  : Noted.   [wrote long-term memory]

  Student: My favourite subject is machine learning.
  Agent  : Noted.   [wrote long-term memory]

  Student: What is my favourite subject?
  Agent  : My favourite subject is machine learning.   [long-term memory, short-term memory]

  Student: What is 45 * 12?
  Agent  : 540   [tool: calculator]

  Student: Double that.
  Agent  : 1080   [short-term memory, tool: calculator]

  Student: Now subtract 80 from it.
  Agent  : 1000   [short-term memory, tool: calculator]

  Student: Search: what is overfitting?
  Agent  : Overfitting happens when a model learns the training data too well, including its noise, and then performs poorly on new data.   [tool: search]

  Student: Run: print(sum(range(1, 11)))
  Agent

In [5]:
store = VectorStore.from_json(saved)
for q in ["When is my exam?", "What is my favourite food?", "When do I sit for my paper?"]:
    print(f'Query: "{q}"')
    for score, text, seq in store.search(q, k=3):
        flag = "trusted" if score >= RETRIEVAL_MIN else "below threshold"
        print(f"   {score:5.3f}  {text}   ({flag})")
    print()

Query: "When is my exam?"
   0.532  My exam is on 12 December.   (trusted)
   0.444  My exam moved to 15 December.   (trusted)
   0.000  My favourite subject is machine learning.   (below threshold)

Query: "What is my favourite food?"
   0.300  My favourite subject is machine learning.   (below threshold)
   0.000  My exam moved to 15 December.   (below threshold)
   0.000  My exam is on 12 December.   (below threshold)

Query: "When do I sit for my paper?"
   0.000  My favourite subject is machine learning.   (below threshold)
   0.000  My exam moved to 15 December.   (below threshold)
   0.000  My exam is on 12 December.   (below threshold)



In [6]:
# (session, question, expected phrase, category)
TESTS = [
 (1, "What is my favourite subject?",           "machine learning",   "recall within session"),
 (1, "What is 45 * 12?",                        "540",                "tool: calculator"),
 (1, "Double that.",                            "1080",               "short-term reference"),
 (1, "Now subtract 80 from it.",                "1000",               "short-term reference"),
 (1, "Search: what is overfitting?",            "too well",           "tool: search"),
 (1, "Run: print(sum(range(1, 11)))",           "55",                 "tool: code"),
 (1, "When is my exam?",                        "12 December",        "long-term (buffer expired)"),
 (2, "Double that.",                            "I don't know",       "abstain: new session, no antecedent"),
 (2, "When is my exam?",                        "15 December",        "long-term + updated fact"),
 (2, "How many hours do I study per day?",      "3 hours",            "long-term recall"),
 (2, "What project am I working on?",           "spam filter",        "long-term recall"),
 (2, "How many hours will I study in 10 days?", "30 hours",           "memory + tool"),
 (2, "How many days are left until my exam?",   "14 days",            "memory + tool"),
 (2, "What is my mother's phone number?",       "I don't know",       "abstain: never told"),
 (2, "What is my favourite food?",              "I don't know",       "abstain: similar but wrong topic"),
 (2, "When do I sit for my paper?",             "15 December",        "paraphrase (known limitation)"),
 (2, "What is precision?",                      "predicted positives","tool: search"),
]
S1_TURNS = [m for m in SESSION_1]
S2_TURNS = [m for m in SESSION_2]

CONFIGS = {
    "C0 baseline (nothing)":      (False, False, False),
    "C1 short-term only":         (True,  False, False),
    "C2 short + long-term":       (True,  True,  False),
    "C3 tools only":              (False, False, True),
    "C4 full agent":              (True,  True,  True),
}

def run_config(flags):
    """Runs both sessions; returns {(session, question): answer}."""
    a1 = Agent(*flags)
    answers = {}
    for m in S1_TURNS:
        answers[(1, m)] = a1.respond(m)["answer"]
    a2 = Agent(*flags, store_json=a1.store.to_json() if flags[1] else None)    # new session
    for m in S2_TURNS:
        answers[(2, m)] = a2.respond(m)["answer"]
    return answers

def score_all(flags):
    ans = run_config(flags)
    return [(cat, q, exp, ans[(s, q)], exp.lower() in ans[(s, q)].lower()) for s, q, exp, cat in TESTS]

RESULTS = {name: score_all(flags) for name, flags in CONFIGS.items()}

print(f'{"agent version":26} {"correct":8} {"accuracy":9}')
for name, rows in RESULTS.items():
    ok = sum(r[4] for r in rows)
    print(f"{name:26} {ok:>2}/{len(rows):<5} {ok/len(rows):7.0%}   " + "#" * ok)

agent version              correct  accuracy 
C0 baseline (nothing)       3/17        18%   ###
C1 short-term only          4/17        24%   ####
C2 short + long-term        8/17        47%   ########
C3 tools only               7/17        41%   #######
C4 full agent              16/17        94%   ################


In [7]:
names = list(CONFIGS)
short = ["C0", "C1", "C2", "C3", "C4"]
print(f'{"category / question":58}' + " ".join(f"{s:>3}" for s in short))
for i, (s, q, exp, cat) in enumerate(TESTS):
    label = f"{cat}: {q}"[:56]
    print(f"{label:58}" + " ".join(f"{'ok' if RESULTS[n][i][4] else '--':>3}" for n in names))

print("\nWhat the pattern shows:")
cats = {}
for i, (s, q, exp, cat) in enumerate(TESTS):
    cats.setdefault(cat, []).append(i)
for cat, idx in cats.items():
    who = [short[k] for k, n in enumerate(names) if all(RESULTS[n][i][4] for i in idx)]
    print(f"  {cat:36} fully solved by: {', '.join(who) if who else 'nobody'}")

category / question                                        C0  C1  C2  C3  C4
recall within session: What is my favourite subject?       --  ok  ok  --  ok
tool: calculator: What is 45 * 12?                         --  --  --  ok  ok
short-term reference: Double that.                         --  --  --  --  ok
short-term reference: Now subtract 80 from it.             --  --  --  --  ok
tool: search: Search: what is overfitting?                 --  --  --  ok  ok
tool: code: Run: print(sum(range(1, 11)))                  --  --  --  ok  ok
long-term (buffer expired): When is my exam?               --  --  ok  --  ok
abstain: new session, no antecedent: Double that.          ok  ok  ok  ok  ok
long-term + updated fact: When is my exam?                 --  --  ok  --  ok
long-term recall: How many hours do I study per day?       --  --  ok  --  ok
long-term recall: What project am I working on?            --  --  ok  --  ok
memory + tool: How many hours will I study in 10 days?     --  -

In [8]:
print("Full agent (C4) failures:")
for cat, q, exp, got, ok in RESULTS["C4 full agent"]:
    if not ok:
        print(f"  [{cat}] {q}\n     expected '{exp}' but got: {got}\n")
print("Wrong answers that SHOULD have been 'I don't know' (hallucination check):")
for n in names:
    bad = [q for cat, q, exp, got, ok in RESULTS[n] if cat.startswith("abstain") and not ok]
    print(f"  {n:26} {len(bad)} -> {bad}")

Full agent (C4) failures:
  [paraphrase (known limitation)] When do I sit for my paper?
     expected '15 December' but got: I don't know.

Wrong answers that SHOULD have been 'I don't know' (hallucination check):
  C0 baseline (nothing)      0 -> []
  C1 short-term only         0 -> []
  C2 short + long-term       0 -> []
  C3 tools only              0 -> []
  C4 full agent              0 -> []


In [9]:
print(f'{"RETRIEVAL_MIN":14} {"correct":8} wrongly answered (should abstain)   refused but should answer')
old = RETRIEVAL_MIN
for th in (0.05, 0.20, 0.30, 0.35, 0.45, 0.60, 0.80):
    RETRIEVAL_MIN = th
    rows = score_all(CONFIGS["C4 full agent"])
    ok = sum(r[4] for r in rows)
    wrong = sum(1 for r in rows if r[0].startswith("abstain") and not r[4])
    refused = sum(1 for r in rows if (not r[0].startswith("abstain")) and (not r[4]) and "I don't know" in r[3] and not r[0].startswith("paraphrase"))
    print(f"{th:<14} {ok:>2}/{len(rows):<5} {wrong:^34} {refused}")
RETRIEVAL_MIN = old

RETRIEVAL_MIN  correct  wrongly answered (should abstain)   refused but should answer
0.05           15/17                    1                  0
0.2            15/17                    1                  0
0.3            15/17                    1                  0
0.35           16/17                    0                  0
0.45           14/17                    0                  0
0.6            13/17                    0                  3
0.8             9/17                    0                  7


In [10]:
old = BUFFER_SIZE
print(f'{"BUFFER_SIZE":12} "When is my exam?" (turn 11) answered from the buffer?')
for n in (1, 2, 4, 6, 8, 10, 12):
    BUFFER_SIZE = n
    a = Agent(True, False, False)
    out = [a.respond(m)["answer"] for m in S1_TURNS[:11]]
    print(f"{n:<12} {'yes' if '12 december' in out[-1].lower() else 'no'}")
BUFFER_SIZE = old

BUFFER_SIZE  "When is my exam?" (turn 11) answered from the buffer?
1            no
2            no
4            no
6            no
8            no
10           yes
12           yes


In [11]:
for snippet in ["print(2 + 3)", "import os", "open('secret.txt')", "print(2 ** 99999)", "while True: pass", "print(1/0)"]:
    print(f"{snippet:28} -> {run_code(snippet)}")

print(2 + 3)                 -> 5
import os                    -> BLOCKED: code uses something not allowed in this sandbox
open('secret.txt')           -> BLOCKED: code uses something not allowed in this sandbox
print(2 ** 99999)            -> BLOCKED: code uses something not allowed in this sandbox
while True: pass             -> BLOCKED: code uses something not allowed in this sandbox
print(1/0)                   -> ERROR: ZeroDivisionError
